## Open notebook in:
| Colab                                 |  
|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH07/ch07_decision_transformer.ipynb)                                             

# About this notebook

This is a Python 3.12 / Google Colab compatible revision of the original Decision Transformer example.
It replaces `mujoco_py`, `free-mujoco-py`, legacy Gym, and `colabgymrender` with Gymnasium and the official MuJoCo Python bindings.


# Installing dependencies

This version targets Google Colab with Python 3.12.
It uses Gymnasium + the official Google DeepMind `mujoco` Python bindings,
instead of the deprecated `mujoco_py` / `free-mujoco-py` stack.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq \
    libosmesa6-dev \
    ffmpeg


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libpciaccess-dev:amd64.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../0-libpciaccess-dev_0.16-3_amd64.deb ...
Unpacking libpciaccess-dev:amd64 (0.16-3) ...
Selecting previously unselected package libdrm-dev:amd64.
Preparing to unpack .../1-libdrm-dev_2.4.113-2~ubuntu0.22.04.1_amd64.deb ...
Unpacking libdrm-dev:amd64 (2.4.113-2~ubuntu0.22.04.1) ...
Selecting previously unselected package libglx-dev:amd64.
Preparing to unpack .../2-libglx-dev_1.4.0-1_amd64.deb ...
Unpacking libglx-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libgl-dev:amd64.
Preparing to unpack .../3-libgl-dev_1.4.0-1_amd64.deb ...
Unpacking libgl-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package mesa-common-dev:a

## Installing Python packages

Gymnasium v4 MuJoCo environments use the official `mujoco` bindings.
`imageio` is used to write the rollout video without `colabgymrender`.


In [ ]:
!pip -q install \
    "gymnasium[mujoco]==1.3.0" \
    "mujoco==3.10.0" \
    "transformers==4.38.2" \
    "imageio==2.37.0" \
    "imageio-ffmpeg>=0.5.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.8/315.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have

## Importing the packages

`MUJOCO_GL=osmesa` enables headless software rendering on Colab.
It must be set before importing MuJoCo.


In [ ]:
import os

os.environ["MUJOCO_GL"] = "osmesa"

import imageio.v2 as imageio
import gymnasium as gym
import mujoco
import numpy as np
import torch

from IPython.display import Video, display
from transformers import DecisionTransformerModel

import warnings
warnings.filterwarnings("ignore")

print("Python-compatible stack:")
print("  Gymnasium:", gym.__version__)
print("  MuJoCo:", mujoco.__version__)
print("  PyTorch:", torch.__version__)


Python-compatible stack:
  Gymnasium: 1.3.0
  MuJoCo: 3.10.0
  PyTorch: 2.11.0+cpu


## MuJoCo / Gymnasium smoke test

Before downloading the Decision Transformer model, verify that the environment and headless renderer work.


In [ ]:
test_env = gym.make("HalfCheetah-v4", render_mode="rgb_array")
test_obs, test_info = test_env.reset(seed=0)
test_action = test_env.action_space.sample()
test_obs, test_reward, test_terminated, test_truncated, test_info = test_env.step(test_action)
test_frame = test_env.render()

print("Observation shape:", test_obs.shape)
print("Action shape:", test_action.shape)
print("Rendered frame shape:", test_frame.shape)
print("Smoke test: OK")

test_env.close()


Observation shape: (17,)
Action shape: (6,)
Rendered frame shape: (480, 480, 3)
Smoke test: OK


## Defining a function that performs masked autoregressive predictive of actions.

The model's prediction is conditioned on sequences of states, actions, time-steps and returns. The action for the current time-step is included as zeros and masked in to not skew the model's attention distribution.

In [ ]:
def get_action(model, states, actions, rewards, returns_to_go, timesteps):
    max_length = model.config.max_length
    state_dim = model.config.state_dim
    action_dim = model.config.act_dim
    device = states.device

    states = states.reshape(1, -1, state_dim)[:, -max_length:]
    actions = actions.reshape(1, -1, action_dim)[:, -max_length:]
    rewards = rewards.reshape(1, -1, 1)[:, -max_length:]
    returns_to_go = returns_to_go.reshape(1, -1, 1)[:, -max_length:]
    timesteps = timesteps.reshape(1, -1)[:, -max_length:]

    padding = max_length - states.shape[1]

    attention_mask = torch.cat(
        [
            torch.zeros(padding, device=device),
            torch.ones(states.shape[1], device=device),
        ],
        dim=0,
    ).to(dtype=torch.long).reshape(1, -1)

    def pad_tensor(x, dim):
        pad = torch.zeros(
            (1, padding, dim),
            device=device,
            dtype=x.dtype,
        )
        return torch.cat([pad, x], dim=1).float()

    states = pad_tensor(states, state_dim)
    actions = pad_tensor(actions, action_dim)
    rewards = pad_tensor(rewards, 1)
    returns_to_go = pad_tensor(returns_to_go, 1)

    timesteps = torch.cat(
        [
            torch.zeros((1, padding), device=device, dtype=torch.long),
            timesteps,
        ],
        dim=1,
    )

    with torch.inference_mode():
        _, action_preds, _ = model(
            states=states,
            actions=actions,
            rewards=rewards,
            returns_to_go=returns_to_go,
            timesteps=timesteps,
            attention_mask=attention_mask,
            return_dict=False,
        )

    return action_preds[0, -1]


In [ ]:
def interact_with_environment(
    env,
    model,
    state_mean,
    state_std,
    max_ep_len,
    target_return_value,
    state_dim,
    act_dim,
    scale,
    device,
    video_path=None,
    fps=30,
):
    episode_return, episode_length = 0.0, 0

    # Gymnasium API: reset() returns (observation, info)
    state, _ = env.reset(seed=0)

    target_return = torch.tensor(
        [[target_return_value]],
        device=device,
        dtype=torch.float32,
    )

    states = torch.from_numpy(state).reshape(1, state_dim).to(
        device=device, dtype=torch.float32
    )
    actions = torch.zeros((0, act_dim), device=device, dtype=torch.float32)
    rewards = torch.zeros(0, device=device, dtype=torch.float32)
    timesteps = torch.tensor([[0]], device=device, dtype=torch.long)

    writer = None
    if video_path is not None:
        os.makedirs(os.path.dirname(video_path) or ".", exist_ok=True)
        writer = imageio.get_writer(video_path, fps=fps, codec="libx264")
        writer.append_data(env.render())

    try:
        for t in range(max_ep_len):
            actions = torch.cat(
                [actions, torch.zeros((1, act_dim), device=device)],
                dim=0,
            )
            rewards = torch.cat(
                [rewards, torch.zeros(1, device=device)],
                dim=0,
            )

            normalized_states = (states - state_mean) / state_std
            action = get_action(
                model,
                normalized_states,
                actions,
                rewards,
                target_return,
                timesteps,
            )
            actions[-1] = action

            action_np = action.detach().cpu().numpy()
            action_np = np.clip(
                action_np,
                env.action_space.low,
                env.action_space.high,
            )

            # Gymnasium API: step() returns five values
            state, reward, terminated, truncated, _ = env.step(action_np)
            done = terminated or truncated

            cur_state = torch.from_numpy(state).reshape(1, state_dim).to(
                device=device, dtype=torch.float32
            )
            states = torch.cat([states, cur_state], dim=0)
            rewards[-1] = float(reward)

            pred_return = target_return[0, -1] - (float(reward) / scale)
            target_return = torch.cat(
                [target_return, pred_return.reshape(1, 1)],
                dim=1,
            )
            timesteps = torch.cat(
                [
                    timesteps,
                    torch.tensor(
                        [[t + 1]],
                        device=device,
                        dtype=torch.long,
                    ),
                ],
                dim=1,
            )

            episode_return += float(reward)
            episode_length += 1

            if writer is not None:
                writer.append_data(env.render())

            if done:
                break
    finally:
        if writer is not None:
            writer.close()

    return episode_return, episode_length


In [ ]:
def setup_environment(env_name, env, device):
    if env_name == "HalfCheetah-v4":
        state_mean = np.array(
            [-0.06845774, 0.01641455, -0.18354906, -0.27624607, -0.34061527,
             -0.09339716, -0.21321271, -0.08774239, 5.1730075, -0.04275195,
             -0.03610836, 0.14053793, 0.06049833, 0.09550975, 0.067391,
             0.00562739, 0.01338279],
            dtype=np.float32,
        )
        state_std = np.array(
            [0.07472999, 0.30234998, 0.3020731, 0.34417078, 0.17619242,
             0.5072056, 0.25670078, 0.32948127, 1.2574149, 0.7600542,
             1.9800916, 6.5653625, 7.4663677, 4.472223, 10.566964,
             5.6719327, 7.498259],
            dtype=np.float32,
        )
        model_name = "edbeeching/decision-transformer-gym-halfcheetah-medium"

    elif env_name == "Walker2d-v4":
        state_mean = np.array(
            [1.218966, 0.14163373, -0.03704914, -0.1381431, 0.51382244,
             -0.0471911, -0.47288352, 0.04225416, 2.3948874, -0.03143199,
             0.04466356, -0.02390724, -0.10134014, 0.09090938,
             -0.00419264, -0.12120572, -0.5497064],
            dtype=np.float32,
        )
        state_std = np.array(
            [0.12311358, 0.324188, 0.11456084, 0.26230657, 0.5640279,
             0.22718786, 0.38373196, 0.7373677, 1.2387927, 0.7980206,
             1.5664079, 1.8092705, 3.0256042, 4.062486, 1.4586568,
             3.744569, 5.585129],
            dtype=np.float32,
        )
        model_name = "edbeeching/decision-transformer-gym-walker2d-medium"

    else:
        raise ValueError(f"Unsupported environment: {env_name}")

    state_mean = torch.tensor(state_mean, device=device, dtype=torch.float32)
    state_std = torch.tensor(state_std, device=device, dtype=torch.float32)

    model = DecisionTransformerModel.from_pretrained(model_name).to(device)
    model.eval()

    state_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    if state_dim != model.config.state_dim:
        raise RuntimeError(
            f"Observation size mismatch: env={state_dim}, model={model.config.state_dim}"
        )
    if act_dim != model.config.act_dim:
        raise RuntimeError(
            f"Action size mismatch: env={act_dim}, model={model.config.act_dim}"
        )

    return env, model, state_mean, state_std, state_dim, act_dim


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
max_ep_len = 1000
scale = 1000.0
directory = "./video"

print("device:", device)


device: cpu


In [ ]:
# The original Decision Transformer evaluation targets are:
#   HalfCheetah: 12000
#   Walker2d:     5000
env_options = {
    "1": {
        "env_name": "HalfCheetah-v4",
        "target_return": 12000,
    },
    "2": {
        "env_name": "Walker2d-v4",
        "target_return": 5000,
    },
}


In [ ]:
user_input = input(
    "Enter 1 for HalfCheetah-v4 or 2 for Walker2d-v4: "
)


Enter 1 for HalfCheetah-v4 or 2 for Walker2d-v4: 1


In [ ]:
if user_input not in env_options:
    raise ValueError("Invalid input. Please enter 1 or 2.")

selected = env_options[user_input]
env_name = selected["env_name"]
TARGET_RETURN = selected["target_return"] / scale

print("Environment:", env_name)
print("Target return:", selected["target_return"])


Environment: HalfCheetah-v4
Target return: 12000


In [ ]:
# v4 keeps the classic MuJoCo task while using the official `mujoco` bindings.
# rgb_array rendering works in headless Colab via OSMesa.
env = gym.make(env_name, render_mode="rgb_array")

env, model, state_mean, state_std, state_dim, act_dim = setup_environment(
    env_name,
    env,
    device,
)

print("Observation dimension:", state_dim)
print("Action dimension:", act_dim)


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/6.61M [00:00<?, ?B/s]

Observation dimension: 17
Action dimension: 6


In [ ]:
video_path = os.path.join(directory, f"{env_name}.mp4")

episode_return, episode_length = interact_with_environment(
    env=env,
    model=model,
    state_mean=state_mean,
    state_std=state_std,
    max_ep_len=max_ep_len,
    target_return_value=TARGET_RETURN,
    state_dim=state_dim,
    act_dim=act_dim,
    scale=scale,
    device=device,
    video_path=video_path,
    fps=30,
)

env.close()

print(f"Episode return: {episode_return:.2f}")
print(f"Episode length: {episode_length}")
print(f"Video: {video_path}")

display(Video(video_path, embed=True))


Episode return: 5014.46
Episode length: 1000
Video: ./video/HalfCheetah-v4.mp4


In [ ]:
import numpy as np

# Example input embeddings (batch_size, sequence_length, embedding_dim)
input_embeddings = np.random.rand(2, 5, 8)  # 2 sequences, each of length 5, embedding dimension 8

# Example positional embeddings (batch_size, sequence_length, embedding_dim)
positional_embeddings = np.random.rand(2, 5, 8)  # Same dimensions as input embeddings

# Adding positional embeddings
added_embeddings = input_embeddings + positional_embeddings

# Concatenating positional embeddings
concatenated_embeddings = np.concatenate((input_embeddings, positional_embeddings), axis=-1)


print("\nShape of Input Embeddings:", input_embeddings.shape)
print("Shape of Positional Embeddings:", positional_embeddings.shape)
print("Shape of Added Embeddings:", added_embeddings.shape)
print("Shape of Concatenated Embeddings:", concatenated_embeddings.shape)



Shape of Input Embeddings: (2, 5, 8)
Shape of Positional Embeddings: (2, 5, 8)
Shape of Added Embeddings: (2, 5, 8)
Shape of Concatenated Embeddings: (2, 5, 16)
